In [30]:
# IMPORTS
import importlib
import re
import shutil
import numpy as np
import pandas as pd
import nltk
import spacy
import plotly.graph_objects as go
import plotly.io as pio
import pdfplumber

from pathlib import Path
from tqdm import tqdm
from termcolor import colored
from IPython.display import display
from spacy.lang.en.stop_words import STOP_WORDS
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline, AutoTokenizer
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
import kagglehub

import model_nlp

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5049.25it/s]


In [31]:

# Télécharger les datasets
path_jobs = kagglehub.dataset_download("ravindrasinghrana/job-description-dataset")
path_cv = kagglehub.dataset_download("suriyaganesh/resume-dataset-structured")

# Destinations
destination_job = Path("../data").expanduser()
destination_cv = Path("../data").expanduser()

# Copier les datasets
shutil.copytree(path_jobs, destination_job, dirs_exist_ok=True)
shutil.copytree(path_cv, destination_cv, dirs_exist_ok=True)

print("Job dataset copié dans :", destination_job)
print("CV dataset copié dans :", destination_cv)

Job dataset copié dans : ../data
CV dataset copié dans : ../data


In [32]:
import sys
sys.path.append("../modules")

# Prétraitement des jobs descriptions

In [33]:
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /home/onyxia/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [34]:
from preprocessing_job_description import clean_text, build_full_text, enable_tqdm_pandas

## chargement du dataset job-description

In [35]:
jd = pd.read_csv("../data/job_descriptions.csv", on_bad_lines="skip")
print(jd.shape)
jd.head()

(1615940, 23)


,Job Id,Experience,Qualifications,Salary Range,location,Country,latitude,longitude,Work Type,Company Size,...,Contact,Job Title,Role,Job Portal,Job Description,Benefits,skills,Responsibilities,Company,Company Profile
0,1089843540111562,5 to 15 Years,M.Tech,$59K-$99K,Douglas,Isle of Man,54.2361,-4.5481,Intern,26801,...,001-381-930-7517x737,Digital Marketing Specialist,Social Media Manager,Snagajob,Social Media Managers oversee an organizations...,"{'Flexible Spending Accounts (FSAs), Relocatio...","Social media platforms (e.g., Facebook, Twitte...","Manage and grow social media accounts, create ...",Icahn Enterprises,"{""Sector"":""Diversified"",""Industry"":""Diversifie..."
1,398454096642776,2 to 12 Years,BCA,$56K-$116K,Ashgabat,Turkmenistan,38.9697,59.5563,Intern,100340,...,461-509-4216,Web Developer,Frontend Web Developer,Idealist,Frontend Web Developers design and implement u...,"{'Health Insurance, Retirement Plans, Paid Tim...","HTML, CSS, JavaScript Frontend frameworks (e.g...","Design and code user interfaces for websites, ...",PNC Financial Services Group,"{""Sector"":""Financial Services"",""Industry"":""Com..."
2,481640072963533,0 to 12 Years,PhD,$61K-$104K,Macao,"Macao SAR, China",22.1987,113.5439,Temporary,84525,...,9687619505,Operations Manager,Quality Control Manager,Jobs2Careers,Quality Control Managers establish and enforce...,"{'Legal Assistance, Bonuses and Incentive Prog...",Quality control processes and methodologies St...,Establish and enforce quality control standard...,United Services Automobile Assn.,"{""Sector"":""Insurance"",""Industry"":""Insurance: P..."
3,688192671473044,4 to 11 Years,PhD,$65K-$91K,Porto-Novo,Benin,9.3077,2.3158,Full-Time,129896,...,+1-820-643-5431x47576,Network Engineer,Wireless Network Engineer,FlexJobs,"Wireless Network Engineers design, implement, ...","{'Transportation Benefits, Professional Develo...",Wireless network design and architecture Wi-Fi...,"Design, configure, and optimize wireless netwo...",Hess,"{""Sector"":""Energy"",""Industry"":""Mining, Crude-O..."
4,117057806156508,1 to 12 Years,MBA,$64K-$87K,Santiago,Chile,-35.6751,-71.5429,Intern,53944,...,343.975.4702x9340,Event Manager,Conference Manager,Jobs2Careers,A Conference Manager coordinates and manages c...,"{'Flexible Spending Accounts (FSAs), Relocatio...",Event planning Conference logistics Budget man...,Specialize in conference and convention planni...,Cairn Energy,"{""Sector"":""Energy"",""Industry"":""Energy - Oil & ..."


## Construction et nettoyage du dataset

In [36]:
#barre de progression du suivi
enable_tqdm_pandas()
jd["raw_text"] = jd.progress_apply(build_full_text, axis=1)
jd["clean_text"] = jd["raw_text"].progress_apply(clean_text)


100%|██████████| 1615940/1615940 [01:35<00:00, 17001.00it/s]


In [37]:
# vérification
jd[["Job Title", "Job Description", "skills", "Responsibilities", "clean_text"]].head()

,Job Title,Job Description,skills,Responsibilities,clean_text
0,Digital Marketing Specialist,Social Media Managers oversee an organizations...,"Social media platforms (e.g., Facebook, Twitte...","Manage and grow social media accounts, create ...",digital marketing specialist social media mana...
1,Web Developer,Frontend Web Developers design and implement u...,"HTML, CSS, JavaScript Frontend frameworks (e.g...","Design and code user interfaces for websites, ...",web developer frontend web developers design i...
2,Operations Manager,Quality Control Managers establish and enforce...,Quality control processes and methodologies St...,Establish and enforce quality control standard...,operations manager quality control managers es...
3,Network Engineer,"Wireless Network Engineers design, implement, ...",Wireless network design and architecture Wi-Fi...,"Design, configure, and optimize wireless netwo...",network engineer wireless network engineers de...
4,Event Manager,A Conference Manager coordinates and manages c...,Event planning Conference logistics Budget man...,Specialize in conference and convention planni...,event manager conference manager coordinates m...


In [38]:
# sauvegarde du dataset job-description
jd_clean = jd[["Role", "clean_text"]].copy()
output_path = "../data/jd_clean.csv"
jd_clean.to_csv(output_path, index=False)
print(f"Prétraitement terminé ! Fichier sauvegardé : {output_path}")

Prétraitement terminé ! Fichier sauvegardé : ../data/jd_clean.csv


# Prétraitement CV

In [39]:
from preprocessing_cv import ensure_list, build_cv

In [40]:
# le dataset est divisé en plusieurs parties , on charge celles dont on a besoin pour le nlp

# cv
people = pd.read_csv("../data/01_people.csv", dtype={"person_id": "int32"})
selected_ids = set(people["person_id"].tolist())

# abilities
abilities_dict = {}
for chunk in pd.read_csv("../data/02_abilities.csv", chunksize=200000, dtype={"person_id": "int32"}):
    chunk = chunk[chunk["person_id"].isin(selected_ids)]
    for pid, group in chunk.groupby("person_id")["ability"]:
        abilities_dict.setdefault(pid, []).extend(group.tolist())


# skills
skills_dict = {}
for chunk in pd.read_csv("../data/05_person_skills.csv", chunksize=200000, dtype={"person_id": "int32"}):
    chunk = chunk[chunk["person_id"].isin(selected_ids)]
    for pid, group in chunk.groupby("person_id")["skill"]:
        skills_dict.setdefault(pid, []).extend(group.tolist())

# education
edu = pd.read_csv("../data/03_education.csv", dtype={"person_id": "int32"})
edu = edu[edu["person_id"].isin(selected_ids)]
education_dict = {}
for pid, group in edu.groupby("person_id"):
    texts = []
    for _, row in group.iterrows():
        text = f"{row['program']} at {row['institution']} ({row['start_date']}) in {row['location']}"
        texts.append(text)
    education_dict[pid] = texts

# experiences
exp = pd.read_csv("../data/04_experience.csv", dtype={"person_id": "int32"})
exp = exp[exp["person_id"].isin(selected_ids)]
experience_dict = {}
for pid, group in exp.groupby("person_id"):
    texts = []
    for _, row in group.iterrows():
        text = f"{row['title']} at {row['firm']} ({row['start_date']} - {row['end_date']}) in {row['location']}"
        texts.append(text)
    experience_dict[pid] = texts


## Regroupement de toutes les parties dans un dataframe

In [41]:
people["abilities"] = people["person_id"].map(abilities_dict).apply(ensure_list)
people["skills"] = people["person_id"].map(skills_dict).apply(ensure_list)
people["education"] = people["person_id"].map(education_dict).apply(ensure_list)
people["experience"] = people["person_id"].map(experience_dict).apply(ensure_list)


### construction du texte du cv

In [42]:
people["cv_text"] = people.apply(build_cv, axis=1)

### Sauvegarde du fichier

In [43]:
output_path = "../data/cv_54k_reconstructed.csv"
people.to_csv(output_path, index=False)
print(f"Reconstruction terminée ! Fichier sauvegardé : {output_path}")

Reconstruction terminée ! Fichier sauvegardé : ../data/cv_54k_reconstructed.csv


# NLP

In [44]:
# chargement dataset
cv = pd.read_csv("../data/cv_54k_reconstructed.csv")
jd_clean = pd.read_csv("../data/jd_clean.csv")

In [45]:
## on va échantillonner les job-description par rapport aux catégories
TARGET_SIZE = 30000
role_counts = jd_clean["Role"].value_counts()
role_proportions = role_counts / role_counts.sum()
role_samples = (role_proportions * TARGET_SIZE).astype(int)

sampled_jd_list = []
for role, n_samples in role_samples.items():
    subset = jd_clean[jd_clean["Role"] == role]
    n_samples = min(n_samples, len(subset))
    sampled = subset.sample(n_samples, random_state=42)
    sampled_jd_list.append(sampled)

sampled_jd = pd.concat(sampled_jd_list).reset_index(drop=True)
jd = sampled_jd

In [46]:
# conversion en liste de textes
cv_texts = cv["cv_text"].fillna("").tolist()
jd_texts = jd["clean_text"].tolist()

## Vectorisation TF-IDF

In [47]:
cv_texts = cv["cv_text"].fillna("").tolist()
jd_texts = jd["clean_text"].tolist()

all_texts = cv_texts + jd_texts

vectorizer = TfidfVectorizer(
    max_features=50000,
    stop_words="english"
)

tfidf_matrix = vectorizer.fit_transform(all_texts)

cv_tfidf = tfidf_matrix[:len(cv_texts)]
jd_tfidf = tfidf_matrix[len(cv_texts):]

## Récupération des top jd pour un cv

In [48]:
import importlib
import def_top_k

In [49]:
from def_top_k import top_k_tfidf, top_k_cv

example_cv = 0

matches = top_k_tfidf(example_cv, cv_tfidf, jd_tfidf, k=5)

scores = [score for _, score in matches]
moyenne = round(np.mean(scores) * 100, 2)

for idx, score in matches:
    print(f"JD {idx} – score {score:.4f}")
    print(jd.loc[idx, "clean_text"][:200], "...\n")

print(f"\nTF-IDF moyenne top-5 : {moyenne}%")

JD 18914 – score 0.3104
database developer sql database developers design implement maintain relational databases using sql structured query language write queries optimize database performance ensure data integrity security ...

JD 18894 – score 0.3104
database developer sql database developers design implement maintain relational databases using sql structured query language write queries optimize database performance ensure data integrity security ...

JD 18900 – score 0.3104
database developer sql database developers design implement maintain relational databases using sql structured query language write queries optimize database performance ensure data integrity security ...

JD 18878 – score 0.3104
database developer sql database developers design implement maintain relational databases using sql structured query language write queries optimize database performance ensure data integrity security ...

JD 18893 – score 0.3104
database developer sql database developers design implem

In [50]:
# on teste

def extract_text_from_pdf(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text += page.extract_text() or ""
    return text


In [51]:


# JOB DESCRIPTION : l'offre ci-dessous a été prise sur le site job-teaser ensae 
new_jd_raw = """Who's Rollee

Rollee is building the next-gen credit bureau to make financial services accessible for everyone. We work with auto lenders, banks, insurance companies, and many other businesses to streamline their underwriting process. You can be an Uber driver, a Tiktoker or thriving as a freelance consultant; we make your income data accessible, readable and in the proper context for lenders so you get what you deserve.

What's the role

We are looking for a Product Data Scientist Intern to join our team during 6 months.

You’ll sit at the intersection of Product and Data, ensuring that Rollee’s data is accurate, reliable, and truly valuable for our customers
Working with various datasets to extract meaningful information with a business approach
Running business QA on datasets we collect
Structuring rough data into exploitable features available through our API
Developing internal and external business intelligence models
Teaming up with Product, Business and Tech to implement data-driven strategies

➡️ You think you can do all of that?

➡️ You are a passionate person, results-oriented, and looking to work in a fast-growing environment surrounded by talented colleagues from around the world?

➡️ We are interested to know more about you!

The profile we are looking for

Fluent in English (we work 100% in English)
Preferred education background in the quantitative field (mathematics, computer science, AI, data science, statistics)
Strong skills in SQL and Python (nice to have: experience with Git)
Good understanding of business
Solid analytics ability and statistic knowledge
Basic knowledge of Machine Learning algorithms and practical implementation
A strong interest in the Fintech space and financial inclusion
Keen to work with people across multiple teams
Eager to learn and take ownership of topics

Why Rollee

We are an early-stage company with the ambition to transform a whole space
We are backed by prestigious VCs and Business angels (Speedinvest, Seedcamp, 20VC…)
You would work in a hybrid mode at Patchwork Republique in Paris with part of our team - please read our Paris Office Policy
We promote flexible working hours: we focus on results first, expect commitment and trust our peers
We promote diversity and inclusion in our day-to-day: our culture is enriched by everyone's culture
We provide a MacBook on your first day
And multiple other benefits that we can discuss further!

Our hiring process

Cultural interview with our People Manager (30min)
Interview with our Data Scientist & Product Manager (1 hour)
Final conversation with our CEO (15min)
In pursuit of our mission, we firmly assert the importance of aligning all team members with our core values. We invite you to explore our website and reflect on whether you resonate with these values. If you find a connection, we welcome you to apply for the position. We look forward to the opportunity to meet you!
"""

new_jd_text = clean_text(new_jd_raw)


#  CV (PDF) : Nous avons demandé à chatgpt de générer un cv en pdf type data scientist

pdf_path = "../cv_test/cv_english.pdf"
raw_cv_text = extract_text_from_pdf(pdf_path)
new_cv_text = clean_text(raw_cv_text)


## Similarité entre le cv généré et l'offre job-teaser

In [52]:
cv_vec = vectorizer.transform([new_cv_text])
jd_vec = vectorizer.transform([new_jd_text])
score_tfidf = cosine_similarity(cv_vec, jd_vec)[0][0]
similarity_tfidf = round(score_tfidf * 100, 2)

print(f"TF-IDF pairwise score : {similarity_tfidf}%")

# Jauge
fig = go.Figure(go.Indicator(
    domain={'x': [0, 1], 'y': [0, 1]},
    value=similarity_tfidf,
    mode="gauge+number",
    title={'text': "TF-IDF Matching percentage (%)"},
    gauge={
        'axis': {'range': [0, 100]},
        'steps': [
            {'range': [0, 50], 'color': "#FFB6C1"},
            {'range': [50, 70], 'color': "#FFFFE0"},
            {'range': [70, 100], 'color': "#90EE90"}
        ],
        'threshold': {'line': {'color': "red", 'width': 4}, 'thickness': 0.75, 'value': 100}
    }
))
fig.update_layout(width=600, height=400)
fig.show()

# Interprétation
if similarity_tfidf < 50:
    print(colored(" Low chance, need to modify your CV!", "red", attrs=["bold"]))
elif similarity_tfidf < 70:
    print(colored(" Good chance but you can improve further!", "yellow", attrs=["bold"]))
else:
    print(colored(" Excellent! You can submit your CV.", "green", attrs=["bold"]))

TF-IDF pairwise score : 30.2%


 Low chance, need to modify your CV!


## SBERT

In [53]:
# Importation du modèle
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

#  on encode  les datasets 
cv_embeddings = model.encode(cv_texts, batch_size=64, show_progress_bar=True)
jd_embeddings = model.encode(jd_texts, batch_size=64, show_progress_bar=True)
np.save("cv_embeddings.npy", cv_embeddings)
np.save("jd_embeddings.npy", jd_embeddings)


Batches: 100%|██████████| 466/466 [04:03<00:00,  1.92it/s]


In [54]:
# on charge les embeddings
cv_embeddings = np.load("cv_embeddings.npy")
jd_embeddings = np.load("jd_embeddings.npy")

### Similarité entre le cv généré et l'offre job-teaser pour SBERT

In [55]:

# SBERT SIMILARITY – découpage en phrases
cv_sentences = [s.strip() for s in new_cv_text.split(". ") if s]
jd_sentences = [s.strip() for s in new_jd_text.split(". ") if s]

# On encode tous les blocs
cv_embs = model_nlp._sentence_model.encode(cv_sentences, convert_to_tensor=True)
jd_embs = model_nlp._sentence_model.encode(jd_sentences, convert_to_tensor=True)

# Calcul des similarités pairwise
cos_scores = util.cos_sim(cv_embs, jd_embs)  
max_score = cos_scores.max().item()  
score_percent = max_score * 100

print(f"Score de similarité SBERT : {max_score:.4f}")
print(f"Score en % : {score_percent:.2f}%")


# JAUGE

fig = go.Figure(go.Indicator(
    domain={'x': [0, 1], 'y': [0, 1]},
    value=score_percent,
    mode="gauge+number",
    title={'text': "Matching CV ↔ Job (%)"},
    gauge={
        'axis': {'range': [0, 100]},
        'steps': [
            {'range': [0, 50], 'color': "#FFB6C1"},
            {'range': [50, 70], 'color': "#FFFFE0"},
            {'range': [70, 100], 'color': "#90EE90"}
        ],
        'threshold': {
            'line': {'color': "red", 'width': 4},
            'thickness': 0.75,
            'value': 100
        }
    }
))

fig.update_layout(width=600, height=400)
fig.show()


# INTERPRÉTATION

if score_percent < 50:
    print(colored("Low chance, need to modify your CV!", "red", attrs=["bold"]))
elif score_percent < 70:
    print(colored("Good chance but you can improve further!", "yellow", attrs=["bold"]))
else:
    print(colored("Excellent! You can submit your CV.", "green", attrs=["bold"]))

Score de similarité SBERT : 0.6823
Score en % : 68.23%


Good chance but you can improve further!


### Top-K des jobs descriptions pour un cv 

In [56]:
example_cv = 0

cv_vec = cv_embeddings[example_cv].reshape(1, -1)
sims_sbert = cosine_similarity(cv_vec, jd_embeddings)[0]
top5_idx = np.argsort(sims_sbert)[::-1][:5]

scores_sbert = sims_sbert[top5_idx]
moyenne_sbert = round(np.mean(scores_sbert) * 100, 2)

for idx in top5_idx:
    print(f"JD {idx} – score {sims_sbert[idx]:.4f}")
    print(jd.loc[idx, "clean_text"][:200], "...\n")

print(f"\nSBERT moyenne top-5 : {moyenne_sbert}%")

JD 2616 – score 0.6156
systems administrator database administrators manage databases ensuring data integrity security efficient performance design implement maintain databases troubleshooting issues database management sys ...

JD 2626 – score 0.6156
systems administrator database administrators manage databases ensuring data integrity security efficient performance design implement maintain databases troubleshooting issues database management sys ...

JD 2650 – score 0.6156
systems administrator database administrators manage databases ensuring data integrity security efficient performance design implement maintain databases troubleshooting issues database management sys ...

JD 2661 – score 0.6156
systems administrator database administrators manage databases ensuring data integrity security efficient performance design implement maintain databases troubleshooting issues database management sys ...

JD 2633 – score 0.6156
systems administrator database administrators manage database

## DOC2VEC

In [57]:

all_texts_d2v = cv_texts + jd_texts

tagged_corpus = [
    TaggedDocument(words=text.split(), tags=[str(i)])
    for i, text in enumerate(all_texts_d2v)
]

doc2vec_model = Doc2Vec(
    vector_size=300,
    window=10,
    min_count=2,
    workers=4,
    epochs=40,
    dm=1,
    dbow_words=1
)

doc2vec_model.build_vocab(tagged_corpus)
doc2vec_model.train(tagged_corpus, total_examples=doc2vec_model.corpus_count, epochs=doc2vec_model.epochs)
print("Modèle réentraîné !")

Modèle réentraîné !


In [59]:
cv_d2v = np.array([doc2vec_model.dv[str(i)] for i in range(len(cv_texts))])
jd_d2v = np.array([doc2vec_model.dv[str(i + len(cv_texts))] for i in range(len(jd_texts))])

np.save("cv_doc2vec.npy", cv_d2v)
np.save("jd_doc2vec.npy", jd_d2v)
print("Embeddings Doc2Vec sauvegardés.")

Embeddings Doc2Vec sauvegardés.


In [61]:
cv_d2v = np.load("cv_doc2vec.npy")
jd_d2v = np.load("jd_doc2vec.npy")

example_cv = 0
cv_vec = cv_d2v[example_cv].reshape(1, -1)
scores_d2v = cosine_similarity(cv_vec, jd_d2v)[0]
top5_idx = np.argsort(scores_d2v)[::-1][:5]

scores_top5 = scores_d2v[top5_idx]
moyenne_d2v = round(np.mean(scores_top5) * 100, 2)

for idx in top5_idx:
    print(f"JD {idx} – score {scores_d2v[idx]:.4f}")
    print(jd.loc[idx, "clean_text"][:200], "...\n")

print(f"\nDoc2Vec moyenne top-5 : {moyenne_d2v}%")

JD 20929 – score 0.3818
landscape architect develop implement urban planning strategies manage land use ensure sustainable city development urban design planning zoning regulations gis mapping tools plan design urban spaces  ...

JD 6786 – score 0.3818
marketing manager product marketing managers promote market products services create marketing strategies conduct market research develop messaging collaborate crossfunctional teams launch promote pro ...

JD 10702 – score 0.3786
web developer frontend web developers design implement user interfaces websites ensuring visually appealing userfriendly collaborate designers backend developers create seamless web experiences users  ...

JD 15227 – score 0.3744
copywriter seo copywriters specialize creating content optimized search engines incorporate keywords seo best practices improve online visibility search engine rankings web pages articles seo search e ...

JD 22907 – score 0.3721
physical therapist specialize providing physical therapy 

### Similiarité entre le cv généré et l'offre pour Doc2vec

In [62]:

cv_sentences = [s.strip().split() for s in new_cv_text.split(".") if len(s.strip()) > 10]
jd_sentences = [s.strip().split() for s in new_jd_text.split(".") if len(s.strip()) > 10]

cv_vecs = np.array([doc2vec_model.infer_vector(s, epochs=50) for s in cv_sentences])
jd_vecs = np.array([doc2vec_model.infer_vector(s, epochs=50) for s in jd_sentences])

scores = cosine_similarity(cv_vecs, jd_vecs)
score_d2v = scores.max(axis=1).mean()
similarity_d2v = round(float(score_d2v) * 100, 2)

print(f"Score Doc2Vec : {similarity_d2v}%")

# Visualization
fig = go.Figure(go.Indicator(
    domain={'x': [0, 1], 'y': [0, 1]},
    value=similarity_d2v,
    mode="gauge+number",
    title={'text': "Doc2Vec Matching percentage (%)"},
    gauge={
        'axis': {'range': [0, 100]},
        'steps': [
            {'range': [0, 50], 'color': "#FFB6C1"},
            {'range': [50, 70], 'color': "#FFFFE0"},
            {'range': [70, 100], 'color': "#90EE90"}
        ],
        'threshold': {'line': {'color': "red", 'width': 4}, 'thickness': 0.75, 'value': 100}
    }
))

fig.update_layout(width=600, height=400)
fig.show()

# Print notification
if similarity_d2v < 50:
    print(colored("Low chance, need to modify your CV!", "red", attrs=["bold"]))
elif similarity_d2v < 70:
    print(colored("Good chance but you can improve further!", "yellow", attrs=["bold"]))
else:
    print(colored("Excellent! You can submit your CV.", "green", attrs=["bold"]))

Score Doc2Vec : 49.39%


Low chance, need to modify your CV!


# CROSSENCODER

In [63]:



pio.renderers.default = "notebook_connected"


# CHUNKING

def chunk_text(text, chunk_size=80):
    tokens = text.split()
    if len(tokens) == 0:
        return []
    return [" ".join(tokens[i:i + chunk_size]) for i in range(0, len(tokens), chunk_size)]

cv_chunks = chunk_text(new_cv_text)
jd_chunks = chunk_text(new_jd_text)

print(f"CV chunks: {len(cv_chunks)} | JD chunks: {len(jd_chunks)}")

if len(cv_chunks) == 0 or len(jd_chunks) == 0:
    raise ValueError("Chunks vides ; vérifier les textes")

# CROSSENCODER SCORING

scores = []

for cv in cv_chunks:
    for jd in jd_chunks:
        try:
            s = model_nlp.compute_crossencoder_score(cv, jd)
            if s is not None:
                scores.append(float(s))
        except:
            pass

if len(scores) == 0:
    raise ValueError("Aucun score CrossEncoder généré")

scores = np.array(scores)

#  SCORE FINAL 
score_raw = float(np.max(scores))
if score_raw < 0:  
    score_percent = (score_raw + 1) / 2 * 100
else:
    score_percent = score_raw * 100

score_percent = float(np.clip(score_percent, 0, 100))

print(f"\nScore brut CrossEncoder: {score_raw:.4f}")
print(f"Score final (%)        : {score_percent:.2f}%")


if score_percent < 50:
    text = "Low match – CV needs improvement"
    color = "red"
elif score_percent < 70:
    text = "Medium match – decent but improvable"
    color = "yellow"
else:
    text = "Strong match – good fit"

print(colored(text, color, attrs=["bold"]))

# JAUGE PLOTLY

fig = go.Figure(go.Indicator(
    domain={'x': [0, 1], 'y': [0, 1]},
    value=score_percent,
    mode="gauge+number",
    title={'text': "CV and Job Matching Score"},
    gauge={
        'axis': {'range': [0, 100]},
        'steps': [
            {'range': [0, 50], 'color': "#ffb6b6"},
            {'range': [50, 70], 'color': "#fff3b0"},
            {'range': [70, 100], 'color': "#b6fcb6"}
        ],
        'threshold': {
            'line': {'color': "red", 'width': 3},
            'thickness': 0.75,
            'value': 100
        }
    }
))

fig.update_layout(width=600, height=400)

display(fig)

CV chunks: 4 | JD chunks: 4

Score brut CrossEncoder: 0.6059
Score final (%)        : 60.59%
Medium match – decent but improvable


## NER

In [65]:
nltk.download("punkt", quiet=True)


# MODEL
ner_pipe = pipeline(
    "ner",
    model="feliponi/hirly-ner-multi",
    aggregation_strategy="simple"
)
tokenizer = AutoTokenizer.from_pretrained("feliponi/hirly-ner-multi")
sentence_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
nlp = spacy.load("en_core_web_sm")
stemmer = PorterStemmer()


# NORMALISATION
def normalize_skill(skill: str) -> str:
    doc = nlp(skill.lower().strip())
    tokens = [t.lemma_ for t in doc if t.text not in STOP_WORDS and not t.is_punct]
    result = " ".join(tokens)
    return result if result else skill.lower().strip()

def stem_skill(skill: str) -> str:
    return " ".join(stemmer.stem(w) for w in skill.lower().split())


def split_into_chunks(text: str, max_tokens: int = 512):
    sentences = [s.strip() for s in text.split("\n") if s.strip()]
    chunks, current_chunk, current_len = [], [], 0

    for sentence in sentences:
        token_len = len(tokenizer.tokenize(sentence))
        if current_len + token_len > max_tokens:
            if current_chunk:
                chunks.append(" ".join(current_chunk))
            current_chunk = [sentence]
            current_len = token_len
        else:
            current_chunk.append(sentence)
            current_len += token_len

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks


# EXTRACTION

VALID_LABELS = {"SKILL", "LANG"}  

def extract_skills(text: str, confidence: float = 0.7) -> dict:
    chunks = split_into_chunks(str(text))
    skills = {}

    for chunk in chunks:
        entities = ner_pipe(chunk)
        for e in entities:
            if e["entity_group"] in VALID_LABELS and e["score"] >= confidence:
                word = e["word"].strip()
                if word:
                    skills[normalize_skill(word)] = word

    return skills


# COMPARAISON 
def compare_skills(cv_skills: dict, jd_skills: dict, semantic_threshold: float = 0.75) -> dict:
    cv_set = {s.lower().strip() for s in cv_skills.keys() if s.strip()}
    jd_set = {s.lower().strip() for s in jd_skills.keys() if s.strip()}

    # Pass 1 : stem matching
    cv_stemmed = {stem_skill(s): s for s in cv_set}
    jd_stemmed = {stem_skill(s): s for s in jd_set}

    present_stems = set(cv_stemmed) & set(jd_stemmed)
    unmatched_jd_stems = set(jd_stemmed) - present_stems

    matched_skills = [
        {"skill": jd_skills[jd_stemmed[s]], "status": "match"}
        for s in present_stems
    ]
    unmatched_jd = [jd_stemmed[s] for s in unmatched_jd_stems]

    # Pass 2 : semantic matching sur mots ORIGINAUX
    truly_missing = []

    if unmatched_jd and cv_set:
        cv_original = list(cv_skills.values())
        cv_embeddings = sentence_model.encode(cv_original)

        for jd_skill in unmatched_jd:
            jd_original = jd_skills.get(jd_skill, jd_skill)
            jd_embedding = sentence_model.encode([jd_original])
            sims = cosine_similarity(jd_embedding, cv_embeddings)[0]
            best_idx = int(np.argmax(sims))
            best_score = float(sims[best_idx])

            if best_score >= semantic_threshold:
                matched_skills.append({
                    "skill": jd_original,
                    "status": "partial"
                })
            else:
                truly_missing.append(jd_original)
    else:
        truly_missing = [jd_skills.get(s, s) for s in unmatched_jd]

    return {
        "cv_skills": sorted(cv_skills.values()),
        "jd_skills": sorted(jd_skills.values()),
        "matched_skills": sorted(matched_skills, key=lambda x: (x["status"], x["skill"])),
        "missing": sorted(truly_missing)
    }


# PIPELINE

cv_skills = extract_skills(new_cv_text)
jd_skills = extract_skills(new_jd_text)

print("CV skills:", sorted(cv_skills.values()))
print("JD skills:", sorted(jd_skills.values()))

result = compare_skills(cv_skills, jd_skills)

print("\n Matched skills:", result["matched_skills"])
print("\n Missing skills:", result["missing"])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2979.87it/s]


CV skills: ['ai', 'algorithms', 'analytics', 'api', 'api integration', 'automation', 'business intelligence', 'computer science', 'dashboards', 'data accuracy', 'data analysis', 'data analyst', 'data processing', 'data science', 'data scientist', 'data validation', 'datadriven', 'designing', 'english', 'french', 'git', 'kpi', 'kpi analysis', 'languages', 'lightgbm', 'machine learning', 'management', 'mathematics', 'metrics', 'nlp', 'numpy', 'pandas', 'power bi', 'predictive modeling', 'programming', 'prototyping', 'python', 'qa', 'reliability', 'sql', 'sql queries', 'statistics', 'structured', 'tableau', 'teams', 'tensorflow']
JD skills: ['ai', 'algorithms', 'ambition', 'analytics', 'api', 'benefits', 'business intelligence', 'computer science', 'consultant', 'data science', 'data scientist', 'datadriven', 'english', 'financial services', 'git', 'hiring', 'implementation', 'insurance', 'machine learning', 'mathematics', 'office', 'product manager', 'python', 'qa', 'read', 'reliable', '